# PayGuard — AI-Powered Payment Fraud Detection

## Notebook 01: Exploratory Data Analysis

This notebook explores the Kaggle Credit Card Fraud Detection dataset.  
The goal is to understand transaction patterns, class imbalance, fraud behavior, and feature relationships before building machine learning models.

### Dataset Columns

- `Time`: Seconds elapsed between each transaction and the first transaction in the dataset
- `V1` to `V28`: PCA-transformed anonymized transaction features
- `Amount`: Transaction amount
- `Class`: Target variable where `0 = legitimate` and `1 = fraud`

## 1. SETUP

In this section, we import the required libraries, configure notebook display settings, load the dataset, and inspect the basic structure of the data.

In [ ]:
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.4f}".format)

PROJECT_ROOT = Path.cwd()

possible_paths: List[Path] = [
    PROJECT_ROOT / "data" / "raw" / "creditcard.csv",
    PROJECT_ROOT.parent / "data" / "raw" / "creditcard.csv",
    Path("../data/raw/creditcard.csv"),
    Path("data/raw/creditcard.csv"),
]

DATA_PATH = next((path for path in possible_paths if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "creditcard.csv was not found. Please place the dataset in "
        "'data/raw/creditcard.csv' and run the notebook again."
    )

df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded successfully from: {DATA_PATH}")
print(f"Dataset shape: {df.shape[0]:,} rows and {df.shape[1]:,} columns")

display(df.dtypes.to_frame(name="Data Type"))

## 2. DATA OVERVIEW

Before analyzing fraud patterns, we first inspect the dataset structure, summary statistics, missing values, duplicate records, and a small sample of transactions.

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
null_summary = (
    df.isnull()
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_values"})
)

null_summary["missing_percentage"] = (
    null_summary["missing_values"] / len(df) * 100
).round(4)

display(null_summary.sort_values("missing_values", ascending=False))

duplicate_count = df.duplicated().sum()
duplicate_percentage = duplicate_count / len(df) * 100

print(f"Duplicate rows: {duplicate_count:,}")
print(f"Duplicate percentage: {duplicate_percentage:.4f}%")

In [ ]:
df.head()

## 3. CLASS DISTRIBUTION

Fraud detection datasets are usually highly imbalanced.  
Here, we compare legitimate and fraudulent transactions by count and percentage.

In [ ]:
class_counts = (
    df["Class"]
    .value_counts()
    .rename(index={0: "Legitimate", 1: "Fraud"})
    .reset_index()
)

class_counts.columns = ["transaction_type", "count"]
class_counts["percentage"] = class_counts["count"] / len(df) * 100

display(class_counts)

fraud_percentage = class_counts.loc[
    class_counts["transaction_type"] == "Fraud", "percentage"
].iloc[0]

print(
    f"Only {fraud_percentage:.4f}% of transactions are fraudulent — severe class imbalance"
)

In [ ]:
fig = px.pie(
    class_counts,
    names="transaction_type",
    values="count",
    title="Transaction Class Distribution",
    hole=0.45,
)

fig.update_traces(
    textposition="inside",
    textinfo="percent+label",
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
)

fig.show()

In [ ]:
fig = px.bar(
    class_counts,
    x="transaction_type",
    y="count",
    text="count",
    title="Legitimate vs Fraud Transaction Count",
    labels={
        "transaction_type": "Transaction Type",
        "count": "Number of Transactions",
    },
)

fig.update_traces(texttemplate="%{text:,}", textposition="outside")

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    yaxis_type="log",
)

fig.show()

## 4. TRANSACTION AMOUNT ANALYSIS

Transaction amount can reveal useful fraud patterns.  
In this section, we compare the amount distribution for legitimate and fraudulent transactions using histograms, box plots, and summary statistics.

In [ ]:
amount_summary = (
    df.groupby("Class")["Amount"]
    .describe()
    .rename(index={0: "Legitimate", 1: "Fraud"})
)

display(amount_summary)

In [ ]:
plot_df = df.copy()
plot_df["Transaction Type"] = plot_df["Class"].map({0: "Legitimate", 1: "Fraud"})

fig = px.histogram(
    plot_df,
    x="Amount",
    color="Transaction Type",
    nbins=100,
    barmode="overlay",
    marginal="rug",
    title="Transaction Amount Distribution: Fraud vs Legitimate",
    labels={"Amount": "Transaction Amount"},
)

fig.update_traces(opacity=0.65)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_range=[0, plot_df["Amount"].quantile(0.99)],
    yaxis_type="log",
)

fig.show()

In [ ]:
fig = px.box(
    plot_df,
    x="Transaction Type",
    y="Amount",
    points="outliers",
    title="Transaction Amount Box Plot: Fraud vs Legitimate",
    labels={
        "Transaction Type": "Transaction Type",
        "Amount": "Transaction Amount",
    },
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    yaxis_range=[0, plot_df["Amount"].quantile(0.99)],
)

fig.show()

In [ ]:
fraud_df = df[df["Class"] == 1].copy()

amount_bins = pd.cut(
    fraud_df["Amount"],
    bins=[0, 1, 10, 25, 50, 100, 250, 500, 1000, np.inf],
    right=False,
    include_lowest=True,
)

fraud_amount_ranges = (
    fraud_df.assign(amount_range=amount_bins)
    .groupby("amount_range", observed=False)
    .size()
    .reset_index(name="fraud_count")
)

fraud_amount_ranges["percentage"] = (
    fraud_amount_ranges["fraud_count"] / fraud_amount_ranges["fraud_count"].sum() * 100
).round(2)

display(fraud_amount_ranges)

top_amount_range = fraud_amount_ranges.sort_values(
    "fraud_count", ascending=False
).iloc[0]

print(
    "Key finding: Most fraudulent transactions are concentrated in the "
    f"{top_amount_range['amount_range']} amount range, representing "
    f"{top_amount_range['percentage']:.2f}% of all fraud cases."
)

In [ ]:
fig = px.bar(
    fraud_amount_ranges,
    x=fraud_amount_ranges["amount_range"].astype(str),
    y="fraud_count",
    text="fraud_count",
    title="Fraud Count by Transaction Amount Range",
    labels={
        "x": "Amount Range",
        "fraud_count": "Number of Fraud Transactions",
    },
)

fig.update_traces(textposition="outside")

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
)

fig.show()

## 5. TIME ANALYSIS

The `Time` column is measured in seconds from the first transaction.  
To make it easier to understand transaction behavior, we convert it into hours and analyze fraud frequency by hour of day.

In [ ]:
time_df = df.copy()
time_df["Hour"] = (time_df["Time"] // 3600) % 24
time_df["Hour"] = time_df["Hour"].astype(int)
time_df["Elapsed_Hour"] = (time_df["Time"] // 3600).astype(int)
time_df["Transaction Type"] = time_df["Class"].map({0: "Legitimate", 1: "Fraud"})

hourly_summary = (
    time_df.groupby("Hour")
    .agg(
        total_transactions=("Class", "count"),
        fraud_transactions=("Class", "sum"),
    )
    .reset_index()
)

hourly_summary["fraud_rate"] = (
    hourly_summary["fraud_transactions"] / hourly_summary["total_transactions"] * 100
)

display(hourly_summary)

In [ ]:
fig = px.line(
    hourly_summary,
    x="Hour",
    y="fraud_rate",
    markers=True,
    title="Fraud Rate by Hour of Day",
    labels={
        "Hour": "Hour of Day",
        "fraud_rate": "Fraud Rate (%)",
    },
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis=dict(dtick=1),
)

fig.show()

In [ ]:
fraud_over_time = (
    time_df[time_df["Class"] == 1]
    .groupby("Elapsed_Hour")
    .size()
    .reset_index(name="fraud_count")
)

fig = px.line(
    fraud_over_time,
    x="Elapsed_Hour",
    y="fraud_count",
    markers=True,
    title="Fraud Frequency Over Elapsed Time",
    labels={
        "Elapsed_Hour": "Elapsed Hour Since First Transaction",
        "fraud_count": "Fraud Transaction Count",
    },
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
)

fig.show()

In [ ]:
fraud_hour_heatmap = hourly_summary[["Hour", "fraud_transactions"]].copy()
fraud_hour_heatmap["day_group"] = "All Transactions"

heatmap_matrix = fraud_hour_heatmap.pivot(
    index="day_group",
    columns="Hour",
    values="fraud_transactions",
)

fig = px.imshow(
    heatmap_matrix,
    text_auto=True,
    aspect="auto",
    title="Heatmap of Fraud Transactions by Hour",
    labels={
        "x": "Hour of Day",
        "y": "",
        "color": "Fraud Count",
    },
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
)

fig.show()

## 6. FEATURE CORRELATIONS

The `V1` to `V28` features are anonymized PCA-transformed variables.  
Even though their original business meaning is hidden, correlation analysis can help identify which features have stronger relationships with fraud.

In [ ]:
correlations = (
    df.corr(numeric_only=True)["Class"]
    .drop("Class")
    .sort_values(key=lambda values: values.abs(), ascending=False)
)

top_10_features = correlations.head(10)

display(
    top_10_features
    .reset_index()
    .rename(columns={"index": "feature", "Class": "correlation_with_class"})
)

In [ ]:
top_corr_features = top_10_features.index.tolist() + ["Class"]
corr_matrix = df[top_corr_features].corr(numeric_only=True)

fig = px.imshow(
    corr_matrix,
    text_auto=".2f",
    aspect="auto",
    title="Correlation Heatmap: Top 10 Features Most Correlated with Fraud Class",
    labels={
        "color": "Correlation",
    },
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
)

fig.show()

In [ ]:
positive_features = top_10_features[top_10_features > 0]
negative_features = top_10_features[top_10_features < 0]

print("Most predictive V features based on absolute correlation with Class:")
for feature, value in top_10_features.items():
    direction = "positive" if value > 0 else "negative"
    print(f"- {feature}: {value:.4f} ({direction} relationship)")

print("\nInterpretation:")
print(
    "Features with stronger absolute correlation values are more useful initial "
    "signals for detecting fraud. However, correlation alone is not enough; "
    "model-based feature importance and SHAP analysis should be used later."
)

## 7. KEY INSIGHTS SUMMARY

Based on the exploratory analysis, the main findings are:

- The dataset has a severe class imbalance, with fraudulent transactions representing only a very small percentage of total transactions.
- Transaction amount patterns differ between fraud and legitimate cases, and most fraud cases are concentrated within specific lower-to-mid amount ranges.
- Fraud activity varies by hour, meaning time-based features may add useful signal to the model.
- The anonymized PCA features `V1` to `V28` contain important fraud signals, with some features showing stronger correlation with the target variable.
- Because fraud is rare, model evaluation should focus on fraud-sensitive metrics such as recall, precision, F1-score, ROC-AUC, and PR-AUC instead of accuracy alone.

## Next Step

The next notebook should focus on data preprocessing, feature engineering, train-test splitting, class imbalance handling, and saving a cleaned dataset for modeling.